# CS1 - Full Formation Evaluation of a North Sea Discovery Well (30/6-5)

**Petrophys Case Studies - Project 1**

This is not a single-module tutorial. It is a realistic, end-to-end *interpretation project*
with a concrete deliverable: a complete formation evaluation of a discovery well and a
zone-by-zone recommendation of **where to perforate**.

We follow the standard case-study spine:

1. **The brief** - the business question and why it matters.
2. **The well and data** - what we have, what is missing.
3. **Workflow plan** - the chain of interpretation steps and why in this order.
4. **Execution** - the dynamic workflow run, with its step trace shown as an audit.
5. **Integrated result** - a multi-track interpretation plot and a per-formation results table.
6. **Interpretation and deliverable** - what the numbers mean, and the ranked completion picks.
7. **Limitations and assumptions** - honest caveats.
8. **Appendix** - how to re-run on another well.

> This lesson *uses* the library end-to-end and references the Part A / Part B tutorials for
> the underlying physics (Vcl from GR, density-neutron porosity, Archie saturation, net-pay
> cutoffs). It does not re-teach those basics.

## Part 1 - The brief

> **From:** Subsurface team lead, North Sea development asset.
> **Subject:** Well 30/6-5 - full formation evaluation and completion recommendation.
>
> We have just finished logging discovery well **30/6-5** (Oseberg area, Norwegian North Sea).
> Wireline is in. Before we design the completion, I need a **defensible, auditable formation
> evaluation** of the whole logged section and a **ranked list of intervals to perforate**.
> Tell me which zones are pay, how good they are (porosity, water saturation, permeability),
> and where you would put the guns first. Flag anything you had to assume.

**Why it matters.** The completion decision commits real money. Perforate the wrong zone and you
produce water or nothing; miss a good zone and you leave reserves behind. The recommendation has
to be traceable - every cutoff and every assumed parameter must be visible so the reservoir
engineer can challenge it. That auditability is exactly what the *dynamic workflow* trace gives us.

**The deliverable (three artifacts):**

- **(a)** a multi-track interpretation / log plot summarizing the evaluation;
- **(b)** a per-formation results table (net pay, average PHIE, average Sw, hydrocarbon pore
  thickness);
- **(c)** a ranked list of candidate completion intervals with written rationale.

## Part 2 - The well and the data

**Well 30/6-5** is a Geolink clastic well from the Oseberg area, logged in **metres**. It penetrates
the full Norwegian North Sea post-Jurassic-to-Statfjord stratigraphy - from the shallow Nordland
shales down through the Brent-equivalent **Baat Group** (Ness / Etive sands) and into the
**Statfjord Formation** near total depth.

What we have and what we are missing drives every modelling choice below, so we inventory it first.

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import petrophys
from petrophys.io import read_las
from petrophys.preprocessing.standardization import standardize_curves
from petrophys.datasets import DATA_DIR

print("petrophys", petrophys.__version__)

def ascii_safe(s):
    # 30/6-5 header field name has mojibake; keep console output ASCII-only.
    return str(s).encode("ascii", "replace").decode("ascii")

# --- Load the raw LAS and standardize curve names / units --------------------
las_path = petrophys.sample("30_6-5")
raw = petrophys.read(las_path)
well = read_las(las_path)
well = standardize_curves(well, rename_curves=True, convert_units=True)

print("Well name      :", well.name)
print("Field          :", ascii_safe(well.header.field))
print("Depth interval :", round(well.start_depth, 1), "-", round(well.stop_depth, 1),
      well.depth_unit)
print("Samples        :", well.n_samples, "at step", round(well.step, 4), well.depth_unit)
print("Curves         :", ", ".join(well.curves))

### A first plot is independent of this interpretation study

Before applying the study's assumptions, inspect the raw LAS. Formations and lithology are optional. This case deliberately keeps the embedded vendor lithology for its interpretation comparison; the companion formation table is used for annotation and summaries.

Attaching external lithology with `petrophys.read(..., lithology=...)` would change the default selected source. That is a meaningful model change, not just plot styling, and requires rechecking the results. The [own-data tutorial](../own_data.ipynb) demonstrates that route.

In [ ]:
raw_viewer = petrophys.plot(raw, depth_range=(2800, 3000), figsize=(15, 8))
raw_viewer.show()
raw_viewer.close()
print("Core saturation display warnings:", raw_viewer.warnings)

### Data readiness check

We split the available curves into the log families the interpretation needs, and note the gaps.
This LAS release carries the operator's **parameter block** in the header (Rw, Rmf, BHT, mud
filtrate chlorides), so the environment does not have to be invented - though Part 3 shows one of
those values needs care before use. The most consequential *data* gap is the photoelectric factor
(PEF), which helps discriminate lithology but is only recorded over a narrow deep interval. Both
facts are revisited in Part 7.

In [ ]:
families = {
    "Gamma ray / clay":   ["GR", "SP"],
    "Porosity (D/N/S)":   ["RHOB", "NPHI", "DTC"],
    "Resistivity":        ["RDEP", "RMED", "RSHA"],
    "Lithology aids":     ["PEF", "DRHO", "LITHOLOGY_GEOLINK"],
    "Borehole / caliper": ["CALI", "BS"],
    "Core":               ["PHIC", "KH_CORE", "SWC", "SOC"],
}
for fam, mnems in families.items():
    present = [m for m in mnems if well.has_curve(m)]
    missing = [m for m in mnems if not well.has_curve(m)]
    print("{:22s} present: {:35s} missing: {}".format(
        fam, ", ".join(present) if present else "-",
        ", ".join(missing) if missing else "-"))

print()
# PEF live interval - it is only recorded over the deep section.
pef = well.get("PEF").data
depth = well.depth
pef_live = depth[np.isfinite(pef)]
print("PEF live interval : {:.0f} - {:.0f} m (of {:.0f} - {:.0f} m logged)".format(
    pef_live.min(), pef_live.max(), depth.min(), depth.max()))

print("Header ~Params block:")
for k in ("RW", "RMF", "BHT", "CLFL"):
    v = well.get_parameter(k)
    u = well.get_parameter(k + "__UNIT") or ""
    d = well.get_parameter(k + "__DESCR") or ""
    print("   {:5s}= {:>10} {:6s} {}".format(k, v, u, d))

## Part 3 - The workflow plan

We use the library's **dynamic workflow** (`run_dynamic_workflow`) as the interpretation backbone.
The dynamic workflow inspects the curves that are present and runs only the modules it can support,
in the correct petrophysical order, recording a **step-by-step trace** we will use as the audit.

The chain it will execute for 30/6-5 (each step references the tutorial that teaches it):

| # | Step | What it does | Key inputs |
|---|------|--------------|-----------|
| 1 | Temperature | Builds a T profile from surface T and BHT (assumed here) | depth, BHT |
| 2 | Bad-hole | Flags washed-out / rugose hole from CALI vs BS | CALI, BS, DRHO |
| 3 | Vcl | Volume of clay from GR (and SP) | GR, SP |
| 4 | Porosity | Density-neutron total porosity PHIT and effective PHIE | RHOB, NPHI |
| 5 | Lithology | Multi-mineral solve (quartz / calcite / dolomite / clay) | RHOB, NPHI, PEF |
| 6 | Core correction | Ties log porosity/perm to core (PHIC, KH_CORE) | PHIC, KH_CORE |
| 7 | Rw | Formation-water resistivity (assumed Rw, temp-corrected) | assumed Rw + T |
| 8 | Saturation | Water saturation Sw (Archie family, auto-routed) | RDEP, PHIE, Rw |
| 9 | Permeability | Log permeability, core-calibrated where possible | PHIE, core |
| 10 | Rock quality | RQI / FZI flow indicators | PHIE, PERM |
| 11 | Net pay | 3-stage cutoffs -> net rock / net reservoir / net pay | Vcl, PHIE, Sw |

The table lists the core interpretation chain; on 30/6-5 the engine actually ran **15 steps**,
picking up the invasion, bulk-volume-water (BVW), moveable-hydrocarbon-index (MHI) and
shale-resistivity (Rcl) extras along the way - all of them appear in the audit trace in Part 4.

We then load the **formation tops** and roll the sample-by-sample answer up to a
**per-formation** summary, and finally turn that into a completion recommendation.

**Model choices and why:**

- **Lithology = clastic, matrix = sandstone.** 30/6-5 is a siliciclastic Jurassic section; the
  reservoirs are the Brent-equivalent and Statfjord sands.
- **Saturation model = auto.** We let the workflow route to the appropriate Archie-family model.
  Case study CS5 does a dedicated Sw-model sensitivity; here we take the workflow's routed answer.
- **Assumed Rw and BHT (flagged).** 30/6-5 has no measured Rw or BHT, so we supply explicit
  assumptions consistent with a moderately saline North Sea Jurassic aquifer, and we carry those
  assumptions into the limitations section. These are the single biggest lever on Sw.

### Parameters: from the header, applied with care

The header's parameter block gives us BHT (126 degC), Rmf (0.08 ohm.m), mud filtrate chlorides
(20,366 mg/l) and a stated **Rw of 0.70 ohm.m** with no temperature basis. That last value cannot
be used at face value: plugged into Archie at reservoir conditions it wets out the entire well
(Sw far above 1 everywhere), which contradicts both the operator's own completion-report
evaluation and the NPD fact page for this well (oil logged from 2841 m down to 2899 m, OWC from
pressure data at 2907 m). Lesson 09 back-calculates the value the operator's evaluation actually
used: **Rw = 0.070 ohm.m at reservoir temperature (~230 F)** - consistent with the header figure
being a report-table printing of that number. We apply 0.070 and record the reconciliation here
and in Part 7.

The surface reference temperature still has to be assumed. Every value below lands in the run
configuration and therefore in the audit trace.

In [ ]:
from petrophys.workflow import run_dynamic_workflow, DynamicWorkflowConfig

# --- Environment from the LAS header (plus two documented decisions) ---------
BHT_F = float(well.get_parameter("BHT")) * 9.0 / 5.0 + 32.0   # 126 degC -> degF

ENV = {
    "surface_temp_f": 50.0,          # ~10 C North Sea seabed - still an assumption
    "bht_f":          round(BHT_F, 1),  # header BHT (126 degC = 258.8 degF)
    "td_ft":          11667.0,       # 3556 m TD expressed in ft for the T gradient
    "rw":             0.070,         # header states 0.70; report-reconciled value
    "rw_meas_temp_f": 230.0,         # Rw basis: reservoir temperature (lesson 09)
}
# --- Interpretation cutoffs: the clastic curriculum standards ----------------
from petrophys.evaluation import STANDARD_CUTOFFS
CUTOFFS = dict(STANDARD_CUTOFFS["clastic"])   # vcl 0.40 / phie 0.08 / sw 0.60

config = DynamicWorkflowConfig(
    lithology="clastic",
    matrix_type="sandstone",
    **ENV,
    **CUTOFFS,
)

print("Environment (header-derived unless noted):")
for k, v in ENV.items():
    print("   {:16s} = {}".format(k, v))
print("Net-pay cutoffs:")
for k, v in CUTOFFS.items():
    print("   {:16s} = {}".format(k, v))

## Part 4 - Execution: run the dynamic workflow and read its audit trace

One call runs the whole chain. The returned `DynamicWorkflowResult` records which steps executed,
which were skipped (and why), the computed Rw, and a full per-step trace. We treat that trace as the
**interpretation audit** - it is what makes the recommendation defensible.

The public `petrophys.interpret` call copies its input. Subsequent calculations use `result.well`; `raw` remains available for QC. The lower-level `run_dynamic_workflow` retains its mutating contract.

In [ ]:
result = petrophys.interpret(well, config=config)
result.raise_for_errors()
well = result.well
assert well is not raw

print("Overall success :", result.success)
print("Steps executed  :", len(result.steps_executed))
print("Steps skipped   :", ", ".join(result.steps_skipped) if result.steps_skipped else "none")

rw_arr = np.asarray(result.rw_computed, dtype=float)
rw_mean = float(np.nanmean(rw_arr)) if rw_arr.size else float("nan")
print("Rw (computed)   : {:.4f} ohm.m (temperature-corrected from {} ohm.m at {} F)".format(
    rw_mean, ENV["rw"], ENV["rw_meas_temp_f"]))

### The step-by-step audit

Below we render each executed step with its method, the curves it consumed, the curves it produced,
and the decisions it logged. Read this top-to-bottom: it is the paper trail from raw logs to net pay.

In [ ]:
for sr in result.steps_executed:
    status = "OK " if sr.success else "FAIL"
    print("[{}] {}".format(status, sr.step))
    if sr.method:
        print("       method : {}".format(sr.method))
    if sr.input_curves:
        print("       inputs : {}".format(", ".join(sr.input_curves)))
    if sr.output_curves:
        print("       outputs: {}".format(", ".join(sr.output_curves)))
    for d in sr.decisions[:4]:
        print("       -> {}".format(ascii_safe(d)))
    if not sr.success and sr.message:
        print("       ERROR: {}".format(ascii_safe(sr.message)))


The same trace is available as a tidy DataFrame (handy for exporting an audit sheet).

In [ ]:
trace_df = result.to_dataframe()
trace_df[["Step", "Status", "Method", "Input Curves", "Output Curves"]]

### QC decision: check where the flag fired

The workflow wrote `NET_PAY`, `PHIE`, `SW` and `PERM` onto the well. Before we trust the numbers
we sanity-check *where* the net-pay flag fired: the depth distribution should be geologically
sensible, and for a discovery well we can hold it against the public record.

In [ ]:
depth = well.depth
netpay = well.get("NET_PAY").data
pay_depths = depth[netpay > 0.5]
step = float(np.nanmedian(np.abs(np.diff(depth))))

print("Net pay flagged: {} samples = {:.1f} m".format(pay_depths.size, pay_depths.size * step))
print("Distribution by depth (200 m bins):")
for lo in range(600, 3600, 200):
    n = int(np.sum((pay_depths >= lo) & (pay_depths < lo + 200)))
    if n:
        print("  {:4d}-{:4d} m : {:4d}  {}".format(lo, lo + 200, n, "#" * min(60, n // 12)))

The distribution is geologically clean, and it matches the public record for this well:

- The pay concentrates in one compact block over the **Brent sands (~2841-2960 m)**. The NPD fact
  page for 30/6-5 reports oil from 2841 m down to 2899 m with an OWC at 2907 m from pressure
  data; our block brackets exactly that.
- A further ~2 m of scattered slivers sits in the **Statfjord** margins (3223-3556 m). The NPD
  records no Statfjord discovery in this well, so we already expect the formation roll-up to show
  these as marginal rather than as a target.
- Nothing books in the shallow section or in the Cretaceous/Paleogene shales.

With the flag distribution verified, we roll up **by named formation** rather than trusting a
single field-wide net-pay total.

## Part 5 - Integrated result

### 5a. Per-formation roll-up

We load the geological tops from `formations_30_6-5.csv` and roll the sample-by-sample interpretation
up to a per-formation summary with `analyze_formations`. This produces net pay, NTG, average PHIE /
Sw / Vcl / permeability, and a hydrocarbon-pore-thickness index (HCPV) for each formation.

In [ ]:
from petrophys.evaluation.summaries.formation_analysis import load_formations_csv, analyze_formations

tops_csv = os.path.join(DATA_DIR, "formations_30_6-5.csv")
formation_table = petrophys.read_formations(tops_csv, well=well)
from petrophys.model.formation_tops import FormationTops
formations = FormationTops.from_dataframe(formation_table).intervals
print("Loaded {} formation intervals from {}".format(len(formations), os.path.basename(tops_csv)))

fa = analyze_formations(well, formations)
fa_df = fa.to_dataframe()

# Deliverable (b): compact per-formation results table.
cols = ["Formation", "Top", "Bottom", "Net Pay (m)", "NTG",
        "Avg PHIE", "Avg Sw", "Avg Perm (mD)", "HCPV"]
table = fa_df[cols].copy()
table = table.round({"Net Pay (m)": 1, "NTG": 3, "Avg PHIE": 3,
                     "Avg Sw": 3, "Avg Perm (mD)": 1, "HCPV": 2})
table

The CSV nests group intervals around their member formations. The per-row figures therefore overlap, but `fa.total_gross` and `fa.total_net_pay` now count the covered depth union once. Do not replace those totals with a sum of all rows.

For the reservoir comparison below, we deliberately select named reservoir members. That geological subset is different from the total across every covered part of the well. HCPV here remains a hydrocarbon pore-thickness index, not volumetric OOIP.

In [ ]:
# Named Jurassic reservoir members we care about (exclude parent groups + shallow artifact).
RESERVOIR_MEMBERS = ["Ness Fm.", "Etive Fm.", "Cook Fm.", "Statfjord Fm."]

res_rows = fa_df[fa_df["Formation"].isin(RESERVOIR_MEMBERS)].copy()
res_rows = res_rows[res_rows["Net Pay (m)"] > 0.5].copy()
res_rows = res_rows.sort_values("HCPV", ascending=False)

show = res_rows[cols].round({"Net Pay (m)": 1, "NTG": 3, "Avg PHIE": 3,
                             "Avg Sw": 3, "Avg Perm (mD)": 1, "HCPV": 2})
print("Reservoir formations carrying net pay (ranked by hydrocarbon pore thickness):")
print(show.to_string(index=False))
print()
print("Whole-well covered-depth union net pay : {:.1f} m".format(fa.total_net_pay))
print("Reservoir-member net pay (no double count)           : {:.1f} m".format(
    res_rows["Net Pay (m)"].sum()))

### 5b. Deliverable (a): the multi-track interpretation plot

We assemble a standard composite interpretation display over the reservoir section
(**~2750-3560 m**), the interval where PEF is live and both real sand packages sit. The
library's "petrophysics" template lays out the full nine-track interpretation panel:

- **Depth** and **lithology** - the depth column plus the Geolink lithology track.
- **GR / Vcl** - gamma ray with the computed clay volume.
- **Resistivity** - deep / medium / shallow curves on a log scale.
- **Neutron / density / sonic** - the raw porosity-tool triplet (NPHI, RHOB, DTC).
- **Porosity** - the porosity family (PHIT / PHIE / PHID / PHIS), the core-corrected PHIE_CC and
  core porosity points (PHIC).
- **Saturation** - Sw with the flushed-zone Sxo, BVW and apparent Swirr, plus core saturation
  points (SWC).
- **Permeability** - PERM / PERM_CC on a log scale against core KH_CORE points.
- **Flags** - the net-rock / net-reservoir / net-pay cutoff flags, shaded.

Green shading in the flags track marks the perforation candidates; formation tops are
annotated automatically.

In [ ]:
from petrophys.visualization import create_log_viewer

TOP, BOT = 2750.0, 3560.0

# Deliverable (a): render the composite interpretation display with the library's
# template-driven log viewer. The "petrophysics" template lays out the standard
# interpretation tracks (GR/Vcl, resistivity on log scale, PHIE/Sw, and the
# NET_ROCK/NET_RES/NET_PAY flag tracks) directly from the curves the workflow
# wrote onto the well. Formation tops are passed as a {name: depth} map so the
# reservoir members and the Baat Gp. top are annotated automatically.
tops = {
    fm.name: fm.top_depth
    for fm in formations
    if TOP <= fm.top_depth <= BOT and fm.name in RESERVOIR_MEMBERS + ["BAAT GP."]
}

viewer = result.plot(
    template_defaults="petrophysics",
    depth_range=(TOP, BOT),
    formation_tops=tops,
)
viewer.fig.suptitle(
    "30/6-5 - Composite interpretation, reservoir section (2750-3560 m)",
    y=1.002, fontsize=12,
)
viewer.show()
viewer.close()
print("Label diagnostics:", viewer.diagnostics["suppressed_formation_labels"])

## Part 6 - Interpretation and the deliverable

### What the numbers mean

Reading the table and the plot together, 30/6-5 has **one strong reservoir package** and little
else - which is exactly what the well's history says:

- **Baat Group - Etive Fm. (2865-2958 m).** Clean, blocky sand: effective porosity ~**21 pu**,
  water saturation ~**0.30**, permeability ~**860 mD**, and **27.7 m** of net pay. This is the
  discovery. The NPD fact page logs oil from 2841 m to 2899 m with the OWC at 2907 m (pressure
  data) - inside this interval: pay above, wet sand below. The operator's 1981 evaluation
  (validated in lesson 09) books 20.75 m in its narrower Etive picks plus 4.25 m in Ness; on
  the modern NPD zonation used here the same rock rolls up to our 27.7 + 5.3 m.
- **Ness Fm. (2841-2865 m).** 5.3 m at Sw ~0.53 - real but thinner pay, closer to the Sw cutoff;
  completed together with Etive, not separately.
- **Statfjord Fm. (3223-3556 m).** Only **2.0 m** of scattered slivers at NTG 0.006 -
  statistically marginal, and consistent with the NPD record, which reports no Statfjord
  discovery in this well. Not a target.
- **Cook Fm.** books nothing.
- **The shallow section books nothing**, and neither do the Cretaceous/Paleogene shales.

### Deliverable (c): ranked candidate completion intervals

We build the ranking directly from the per-formation roll-up, scoring each pay-bearing reservoir
member by hydrocarbon pore thickness (the volume proxy). The output is an ordered perforation
plan with written rationale.

In [ ]:
ranking = res_rows.copy()
ranking = ranking.sort_values("HCPV", ascending=False).reset_index(drop=True)

rationale = {
    "Etive Fm.":     "Clean blocky Baat Gp. sand: Sw ~0.30, ~860 mD - the discovery and primary target.",
    "Ness Fm.":      "Upper Baat sand: thinner and higher Sw - perforate together with Etive.",
    "Statfjord Fm.": "Marginal slivers (NTG ~0.006) in a section with no NPD-recorded discovery.",
    "Cook Fm.":      "Books no pay with header-derived parameters.",
}

print("=" * 92)
print("30/6-5  RANKED COMPLETION CANDIDATES  (perforate top-down)")
print("=" * 92)
print("{:<3s} {:<14s} {:>13s} {:>8s} {:>7s} {:>7s} {:>9s} {:>7s}".format(
    "#", "Formation", "Interval(m)", "NetPay", "PHIE", "Sw", "Perm", "HCPV"))
print("-" * 92)
for i, row in ranking.iterrows():
    print("{:<3d} {:<14s} {:>6.0f}-{:<6.0f} {:>8.1f} {:>7.3f} {:>7.3f} {:>9.1f} {:>7.2f}".format(
        i + 1, row["Formation"], row["Top"], row["Bottom"], row["Net Pay (m)"],
        row["Avg PHIE"], row["Avg Sw"], row["Avg Perm (mD)"], row["HCPV"]))
print("-" * 92)
print()
print("RATIONALE")
for i, row in ranking.iterrows():
    print("  {}. {:<14s} {}".format(i + 1, row["Formation"],
                                     rationale.get(row["Formation"], "")))
print()
# Primary target = the Baat Group Brent sands (Ness + Etive together).
baat = ranking[ranking["Formation"].isin(["Ness Fm.", "Etive Fm."])]
statf = ranking[ranking["Formation"] == "Statfjord Fm."]
print("PRIMARY RECOMMENDATION: perforate the Baat Gp. Brent sands first "
      "(Ness + Etive, {:.0f}-{:.0f} m) - clean, low-Sw, high-perm oil sand.".format(
          baat["Top"].min(), baat["Bottom"].max()))
if not statf.empty:
    print("Statfjord Fm. books only {:.1f} m of marginal slivers (NTG {:.3f}) - "
          "monitor, do not perforate.".format(
              statf["Net Pay (m)"].iloc[0], statf["NTG"].iloc[0]))

**Recommendation in one line:** the **Baat Group (Ness + Etive, 2841-2958 m)** is the completion
target - clean, low-Sw, high-permeability oil sand sitting directly above the NPD's 2907 m OWC;
the **Statfjord Fm.** books only ~2 m of marginal slivers (the NPD records no Statfjord discovery
here) and is not a completion candidate; Cook and the shallow section book nothing. One target,
consistent with both the operator's 1981 evaluation and the NPD fact page.

## Part 7 - Limitations and assumptions

An interpretation is only as trustworthy as its caveats. The material ones for 30/6-5:

- **The header Rw needed reconciliation, not blind trust.** The LAS parameter block states
  `RW = 0.70 ohm.m` with no temperature basis; at face value it wets out the entire well.
  Lesson 09's back-calculation of the operator's evaluation pins the working value at
  **0.070 ohm.m at reservoir temperature**, which we applied (`rw=0.070`, `rw_meas_temp_f=230`).
  Sw scales roughly with `sqrt(Rw)`, so this is the single biggest lever on the answer; a
  dedicated Sw-model / parameter sensitivity is the subject of case study **CS5**.
- **The shallow section is the failure mode to watch.** The unconsolidated Hordaland/Nordland
  interval reads ~40 pu apparent porosity with a mild GR (Vcl ~0.11) and a near-gauge hole, so
  the only cutoff standing between it and a pay flag is Sw. With the applied Rw it reads wet
  and books nothing; an under-estimated Rw would book hundreds of metres of it. Restricting the
  recommendation to named reservoir members - as we did - is cheap insurance.
- **PEF only over a narrow window.** Lithology discrimination is best where PEF is live
  (~2790-3176 m); outside that window the multi-mineral solve leans on density-neutron alone.
  That applies not just to the shallow section but also to most of the Statfjord target
  (3223-3556 m), so the mineral volumes there carry extra uncertainty.
- **Groups and members overlap.** The formation API's aggregate totals use a covered-depth union.
  The members-only reservoir roll-up is an explicit geological subset; do not sum all overlapping rows.
- **Cutoffs are conventions.** `Vcl < 0.40`, `PHIE > 0.08`, `Sw < 0.60` are reasonable clastic
  defaults but are choices; moving them moves the net-pay totals. They are stated explicitly above so
  they can be challenged.
- **HCPV here is a thickness index**, not a volumetric OOIP. It ranks zones per unit area; a true
  volumetric estimate needs area and formation-volume factor (`summation.calculate_ooip`).

## Part 8 - Appendix: reproduce this on another well

The entire evaluation is one workflow call plus one formation roll-up. To re-run on a different well,
swap the sample name, the tops CSV, and (critically) the environment - header parameters where
the LAS carries them (as here), basin-appropriate assumptions where it does not.
The `run_formation_analysis` convenience wrapper does load -> standardize -> workflow -> tops -> roll-up
in a single call; we use it below to reproduce the field-wide roll-up and confirm it matches.

In [ ]:
from petrophys.evaluation.summaries.formation_analysis import run_formation_analysis

fa2 = run_formation_analysis(
    las_path=petrophys.sample("30_6-5"),
    formations_csv_path=os.path.join(DATA_DIR, "formations_30_6-5.csv"),
    lithology="clastic",
    matrix_type="sandstone",
    **ENV,
    **CUTOFFS,
)
print("Convenience wrapper reproduced the analysis.")
print("Total net pay (field-wide) : {:.1f} m".format(fa2.total_net_pay))
print("Formations analysed        :", len(fa2.summaries))


#### Applying this workflow to another well

Start with `petrophys.read` and `petrophys.plot` on the LAS alone. Add optional formation/lithology tables only when available, using the [input templates](../../docs/input_templates.rst). Formation analysis requires bounded intervals or a justified terminal depth.

Then use `petrophys.interpret` with well-specific fluid, temperature and rock assumptions. Inspect the trace, source selection, errors and skipped steps before interpreting the computed curves. The case-study parameters are not universal defaults.

### Recap

- We ran a full, auditable formation evaluation of discovery well 30/6-5 with a single dynamic
  workflow, and used its **step trace** as the interpretation audit.
- We rolled the sample-level answer up **by formation** and produced the three required deliverables:
  a composite interpretation plot, a per-formation results table, and a **ranked completion plan**.
- The verdict: perforate the **Baat Group (Ness/Etive) sands** - the only real target. Statfjord
  books ~2 m of marginal slivers; nothing shallow books.
- The environment came from the **LAS header parameter block** (BHT, Rmf, chlorides) with one
  documented reconciliation: the header's printed RW 0.70 is applied as 0.070 ohm.m at reservoir
  temperature per lesson 09. Case study **CS5** quantifies how much these parameters move the
  answer.

**Next:** CS2 - carbonate reservoir characterization and rock typing on a Costa well.